In [1]:
"""
Примеры кода к Лекции 5.8: Send — динамический параллелизм

Этот модуль демонстрирует:
1. Статический fan-out — параллельные рёбра (для сравнения)
2. Send — базовый пример динамического fan-out
3. Map-Reduce — планировщик → параллельные исследователи → синтезатор
4. Orchestrator-Worker — оркестратор распределяет секции отчёта между воркерами
5. Send через Command — fan-out прямо из узла
6. Гетерогенный fan-out — Send в разные узлы по типу задачи
"""

'\nПримеры кода к Лекции 5.8: Send — динамический параллелизм\n\nЭтот модуль демонстрирует:\n1. Статический fan-out — параллельные рёбра (для сравнения)\n2. Send — базовый пример динамического fan-out\n3. Map-Reduce — планировщик → параллельные исследователи → синтезатор\n4. Orchestrator-Worker — оркестратор распределяет секции отчёта между воркерами\n5. Send через Command — fan-out прямо из узла\n6. Гетерогенный fan-out — Send в разные узлы по типу задачи\n'

In [2]:
import operator
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")

## Статический fan-out (для сравнения)

In [4]:
class SearchState(TypedDict):
    query: str
    findings: Annotated[list[str], operator.add]
    summary: str

def web_search(state: SearchState) -> dict:
    print(f"  [web_search] Ищу в интернете: {state['query']}")
    return {"findings": [f"Веб-результат по '{state['query']}'"]}

def doc_search(state: SearchState) -> dict:
    print(f"  [doc_search] Ищу в документации: {state['query']}")
    return {"findings": [f"Документация по '{state['query']}'"]}

def summarize(state: SearchState) -> dict:
    print(f"  [summarize] Собрано findings: {state['findings']}")
    return {"summary": " + ".join(state["findings"])}

builder = StateGraph(SearchState)
builder.add_node("web_search", web_search)
builder.add_node("doc_search", doc_search)
builder.add_node("summarize", summarize)
# Статический fan-out: два ребра из START
builder.add_edge(START, "web_search")
builder.add_edge(START, "doc_search")
# Fan-in: оба сходятся в summarize
builder.add_edge("web_search", "summarize")
builder.add_edge("doc_search", "summarize")
builder.add_edge("summarize", END)

app = builder.compile()
result = app.invoke({"query": "LangGraph Send API", "findings": [], "summary": ""})
print(f"\n  Итог: {result['summary']}")
print(f"  Число findings: {len(result['findings'])} (всегда 2 — зашито в граф)")

  [doc_search] Ищу в документации: LangGraph Send API  [web_search] Ищу в интернете: LangGraph Send API

  [summarize] Собрано findings: ["Документация по 'LangGraph Send API'", "Веб-результат по 'LangGraph Send API'"]

  Итог: Документация по 'LangGraph Send API' + Веб-результат по 'LangGraph Send API'
  Число findings: 2 (всегда 2 — зашито в граф)


## Send — базовый пример

In [5]:
llm = get_llm()

class JokeState(TypedDict):
    topic: str
    subjects: list[str]
    jokes: Annotated[list[str], operator.add]
    best_joke: str

def generate_topics(state: JokeState) -> dict:
    """Определяет список тем для шуток."""
    response = llm.invoke(
        [
            SystemMessage(
                content="Пользователь даёт тему. Придумай 3 конкретных предмета "
                "для шуток по этой теме. Верни только список через запятую, без нумерации."
            ),
            HumanMessage(content=state["topic"]),
        ]
    )
    subjects = [s.strip() for s in response.content.split(",")]
    print(f"  [generate_topics] Темы: {subjects}")
    return {"subjects": subjects}

def generate_joke(state: dict) -> dict:
    """Генерирует одну шутку. Вызывается через Send для каждой темы."""
    subject = state["subject"]
    response = llm.invoke(
        [
            SystemMessage(
                content="Придумай короткую шутку (1-2 предложения) про указанный предмет."
            ),
            HumanMessage(content=subject),
        ]
    )
    joke = response.content.strip()
    print(f"  [generate_joke] {subject}: {joke[:60]}...")
    return {"jokes": [joke]}

def continue_to_jokes(state: JokeState):
    """Fan-out: создаёт Send для каждой темы."""
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

def best_joke(state: JokeState) -> dict:
    """Reduce: выбирает лучшую шутку из собранных."""
    all_jokes = "\n".join(f"{i + 1}. {j}" for i, j in enumerate(state["jokes"]))
    response = llm.invoke(
        [
            SystemMessage(
                content="Выбери лучшую шутку из списка. Верни только текст шутки."
            ),
            HumanMessage(content=all_jokes),
        ]
    )
    print(f"  [best_joke] Выбрана: {response.content.strip()[:60]}...")
    return {"best_joke": response.content.strip()}

builder = StateGraph(JokeState)
builder.add_node("generate_topics", generate_topics)
builder.add_node("generate_joke", generate_joke)
builder.add_node("best_joke", best_joke)
builder.add_edge(START, "generate_topics")
builder.add_conditional_edges(
    "generate_topics",
    continue_to_jokes,
    ["generate_joke"],
)
builder.add_edge("generate_joke", "best_joke")
builder.add_edge("best_joke", END)

app = builder.compile()
result = app.invoke(
    {"topic": "программирование", "subjects": [], "jokes": [], "best_joke": ""}
)
print(f"\n  Собрано шуток: {len(result['jokes'])}")
print(f"  Лучшая: {result['best_joke']}")

  [generate_topics] Темы: ['баги в коде', 'бесконечные циклы', 'комментарии «потом разберусь»']


  [generate_joke] комментарии «потом разберусь»: Комментарии «потом разберусь» — это как закладка в книге: вр...
  [generate_joke] бесконечные циклы: Бесконечный цикл — это когда программа так увлеклась, что ре...


  [generate_joke] баги в коде: Баги в коде — это как тараканы: даже если один раз прибил, ч...


  [best_joke] Выбрана: Баги в коде — это как тараканы: даже если один раз прибил, ч...

  Собрано шуток: 3
  Лучшая: Баги в коде — это как тараканы: даже если один раз прибил, через минуту уже откуда-то вылез новый.


## Map-Reduce — параллельное исследование

In [6]:
llm = get_llm()

class ResearchState(TypedDict):
    topic: str
    aspects: list[str]
    findings: Annotated[list[str], operator.add]
    report: str

def planner(state: ResearchState) -> dict:
    """Map-фаза: определяет аспекты для исследования."""
    response = llm.invoke(
        [
            SystemMessage(
                content="Пользователь даёт тему для исследования. "
                "Определи 3 ключевых аспекта, которые нужно изучить. "
                "Верни только список через запятую, без нумерации и пояснений."
            ),
            HumanMessage(content=state["topic"]),
        ]
    )
    aspects = [a.strip() for a in response.content.split(",")]
    print(f"  [planner] Аспекты: {aspects}")
    return {"aspects": aspects}

def researcher(state: dict) -> dict:
    """Исследует один аспект. Вызывается через Send для каждого аспекта."""
    aspect = state["aspect"]
    topic = state["topic"]
    response = llm.invoke(
        [
            SystemMessage(
                content="Ты исследователь. Напиши краткий абзац (3-4 предложения) "
                "по указанному аспекту темы."
            ),
            HumanMessage(content=f"Тема: {topic}\nАспект: {aspect}"),
        ]
    )
    finding = f"**{aspect}**: {response.content.strip()}"
    print(f"  [researcher] {aspect}: {response.content.strip()[:60]}...")
    return {"findings": [finding]}

def fan_out_to_researchers(state: ResearchState):
    """Fan-out: Send для каждого аспекта."""
    return [
        Send("researcher", {"aspect": a, "topic": state["topic"]})
        for a in state["aspects"]
    ]

def synthesizer(state: ResearchState) -> dict:
    """Reduce-фаза: объединяет все находки в итоговый отчёт."""
    all_findings = "\n\n".join(state["findings"])
    response = llm.invoke(
        [
            SystemMessage(
                content="На основе собранных данных напиши краткий итоговый отчёт (3-5 предложений). "
                "Объедини ключевые выводы из всех аспектов."
            ),
            HumanMessage(content=f"Собранные данные:\n\n{all_findings}"),
        ]
    )
    print(f"  [synthesizer] Отчёт готов")
    return {"report": response.content.strip()}

builder = StateGraph(ResearchState)
builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("synthesizer", synthesizer)
builder.add_edge(START, "planner")
builder.add_conditional_edges(
    "planner",
    fan_out_to_researchers,
    ["researcher"],
)
builder.add_edge("researcher", "synthesizer")
builder.add_edge("synthesizer", END)

app = builder.compile()
result = app.invoke(
    {
        "topic": "Влияние искусственного интеллекта на рынок труда",
        "aspects": [],
        "findings": [],
        "report": "",
    }
)
print(f"\n  Аспектов исследовано: {len(result['findings'])}")
print(f"\n  Итоговый отчёт:\n  {result['report']}")

  [planner] Аспекты: ['автоматизация и замещение рабочих мест', 'изменение спроса на навыки и профессии', 'влияние на заработные платы и неравенство']


  [researcher] влияние на заработные платы и неравенство: Искусственный интеллект оказывает неоднозначное влияние на з...
  [researcher] автоматизация и замещение рабочих мест: Искусственный интеллект ускоряет автоматизацию многих рутинн...


  [researcher] изменение спроса на навыки и профессии: Искусственный интеллект заметно меняет спрос на навыки и про...


  [synthesizer] Отчёт готов

  Аспектов исследовано: 3

  Итоговый отчёт:
  Искусственный интеллект ускоряет автоматизацию рутинных задач, из-за чего часть рабочих мест в производстве, логистике, сфере услуг и офисной работе постепенно замещается. Одновременно рынок труда перестраивается: растет спрос на специалистов по данным, разработке и обслуживанию ИИ, а также на навыки, которые сложно автоматизировать, — креативность, критическое мышление и коммуникацию. Влияние ИИ на доходы неоднозначно: высококвалифицированные работники могут выигрывать, тогда как зарплаты сотрудников средней и низкой квалификации могут снижаться. В целом ИИ повышает требования к гибкости и переобучению, а без поддержки адаптации может усилить неравенство.


## Orchestrator-worker — генерация отчёта по секциям

In [7]:
llm = get_llm()

class Section(TypedDict):
    name: str
    description: str

class ReportState(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list[str], operator.add]
    final_report: str

def orchestrator(state: ReportState) -> dict:
    """Планирует секции отчёта."""
    response = llm.invoke(
        [
            SystemMessage(
                content="Пользователь даёт тему для отчёта. "
                "Спланируй 3 секции. Для каждой укажи название и краткое описание. "
                "Формат: название: описание (по одной на строку, без нумерации)."
            ),
            HumanMessage(content=state["topic"]),
        ]
    )
    sections = []
    for line in response.content.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            name, desc = line.split(":", 1)
            sections.append({"name": name.strip(), "description": desc.strip()})
    print(f"  [orchestrator] Секции: {[s['name'] for s in sections]}")
    return {"sections": sections}

def worker(state: dict) -> dict:
    """Пишет одну секцию отчёта. Вызывается через Send."""
    section = state["section"]
    topic = state["topic"]
    all_sections = state["all_sections"]
    context = ", ".join(s["name"] for s in all_sections)
    response = llm.invoke(
        [
            SystemMessage(
                content=f"Ты автор отчёта. Напиши секцию '{section['name']}' "
                f"({section['description']}). Другие секции отчёта: {context}. "
                "Пиши связно, 3-5 предложений."
            ),
            HumanMessage(content=f"Тема отчёта: {topic}"),
        ]
    )
    text = f"## {section['name']}\n\n{response.content.strip()}"
    print(f"  [worker] Написана секция: {section['name']}")
    return {"completed_sections": [text]}

def assign_workers(state: ReportState):
    """Fan-out: Send для каждой секции."""
    return [
        Send(
            "worker",
            {
                "section": s,
                "topic": state["topic"],
                "all_sections": state["sections"],
            },
        )
        for s in state["sections"]
    ]

def synthesizer(state: ReportState) -> dict:
    """Собирает секции в итоговый отчёт."""
    report = f"# Отчёт: {state['topic']}\n\n"
    report += "\n\n".join(state["completed_sections"])
    print(
        f"  [synthesizer] Отчёт собран из {len(state['completed_sections'])} секций"
    )
    return {"final_report": report}

builder = StateGraph(ReportState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("worker", worker)
builder.add_node("synthesizer", synthesizer)
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges(
    "orchestrator",
    assign_workers,
    ["worker"],
)
builder.add_edge("worker", "synthesizer")
builder.add_edge("synthesizer", END)

app = builder.compile()
result = app.invoke(
    {
        "topic": "Квантовые вычисления: текущее состояние и перспективы",
        "sections": [],
        "completed_sections": [],
        "final_report": "",
    }
)
print(f"\n  Отчёт ({len(result['completed_sections'])} секций):")
print(f"  {result['final_report'][:200]}...")

  [orchestrator] Секции: ['Введение', 'Текущее состояние', 'Перспективы']


  [worker] Написана секция: Введение


  [worker] Написана секция: Перспективы


  [worker] Написана секция: Текущее состояние
  [synthesizer] Отчёт собран из 3 секций

  Отчёт (3 секций):
  # Отчёт: Квантовые вычисления: текущее состояние и перспективы

## Введение

## Введение

Квантовые вычисления основаны на принципах квантовой механики — суперпозиции, запутанности и интерференции, ко...


## Send через Command — fan-out прямо из узла

In [8]:
class TaskState(TypedDict):
    task: str

class MainState(TypedDict):
    topic: str
    status: str
    results: Annotated[list[str], operator.add]

llm = get_llm()

def planner(state: MainState) -> Command[Literal["worker"]]:
    """Планировщик: обновляет статус И запускает воркеров через Command."""
    response = llm.invoke(
        f"Разбей тему '{state['topic']}' на 3 аспекта для исследования. "
        "Ответь строго 3 строками, по одному аспекту на строку."
    )
    aspects = [line.strip() for line in response.content.strip().split("\n") if line.strip()][:3]
    print(f"  [planner] Аспекты: {aspects}")

    # Command одновременно обновляет состояние и запускает Send
    return Command(
        update={"status": "dispatched"},
        goto=[Send("worker", {"task": aspect}) for aspect in aspects],
    )

def worker(state: TaskState) -> dict:
    """Воркер: исследует один аспект."""
    response = llm.invoke(f"Кратко (1-2 предложения) опиши: {state['task']}")
    print(f"  [worker] {state['task'][:40]}... → готово")
    return {"results": [response.content.strip()]}

builder = StateGraph(MainState)
builder.add_node("planner", planner)
builder.add_node("worker", worker)
builder.add_edge(START, "planner")
# Ребро от worker к END — LangGraph ждёт завершения ВСЕХ Send-вызовов
builder.add_edge("worker", END)

app = builder.compile()
result = app.invoke({"topic": "Искусственный интеллект", "status": "", "results": []})
print(f"\n  Статус: {result['status']}")
print(f"  Результатов: {len(result['results'])}")
for i, r in enumerate(result["results"], 1):
    print(f"  {i}. {r[:80]}...")

  [planner] Аспекты: ['Технологический аспект: алгоритмы, архитектуры моделей, методы обучения и качество данных.', 'Этический аспект: предвзятость, конфиденциальность, прозрачность решений и ответственность.', 'Социально-экономический аспект: влияние на рынок труда, производительность, образование и неравенство.']


  [worker] Социально-экономический аспект: влияние ... → готово
  [worker] Технологический аспект: алгоритмы, архит... → готово


  [worker] Этический аспект: предвзятость, конфиден... → готово

  Статус: dispatched
  Результатов: 3
  1. Технологический аспект включает выбор и настройку алгоритмов и архитектур моделе...
  2. Этический аспект включает необходимость избегать предвзятости в данных и алгорит...
  3. Социально-экономический аспект включает влияние на рынок труда, производительнос...


## Гетерогенный fan-out — Send в разные узлы

In [9]:
llm = get_llm()

class ProjectState(TypedDict):
    goal: str
    deliverables: Annotated[list[str], operator.add]

def manager(state: ProjectState) -> dict:
    """Менеджер анализирует цель и определяет задачи."""
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Ты менеджер проекта. Пользователь описывает цель. "
                    "Определи 2-4 задачи. Каждую пометь типом: research или code. "
                    "Формат строго: тип: описание задачи (по одной на строку)."
                )
            ),
            HumanMessage(content=state["goal"]),
        ]
    )
    tasks = []
    for line in response.content.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            task_type, desc = line.split(":", 1)
            task_type = task_type.strip().lower()
            if "research" in task_type:
                tasks.append({"type": "research", "description": desc.strip()})
            elif "code" in task_type:
                tasks.append({"type": "code", "description": desc.strip()})
    print(f"  [manager] Задачи: {[(t['type'], t['description'][:40]) for t in tasks]}")
    return {"tasks": tasks}

def dispatch_tasks(state: ProjectState):
    """Fan-out в РАЗНЫЕ узлы в зависимости от типа задачи."""
    sends = []
    for task in state.get("tasks", []):
        if task["type"] == "research":
            sends.append(Send("researcher", {"description": task["description"]}))
        elif task["type"] == "code":
            sends.append(Send("coder", {"description": task["description"]}))
    return sends

def researcher(state: dict) -> dict:
    """Исследователь: анализирует и собирает информацию."""
    desc = state["description"]
    response = llm.invoke(
        [
            SystemMessage(content="Ты аналитик. Кратко (2-3 предложения) исследуй тему."),
            HumanMessage(content=desc),
        ]
    )
    result = f"[Исследование] {desc}: {response.content.strip()}"
    print(f"  [researcher] {desc[:40]}...")
    return {"deliverables": [result]}

def coder(state: dict) -> dict:
    """Программист: пишет код или техническое решение."""
    desc = state["description"]
    response = llm.invoke(
        [
            SystemMessage(
                content="Ты программист. Кратко опиши техническое решение (2-3 предложения)."
            ),
            HumanMessage(content=desc),
        ]
    )
    result = f"[Код] {desc}: {response.content.strip()}"
    print(f"  [coder] {desc[:40]}...")
    return {"deliverables": [result]}

def summary(state: ProjectState) -> dict:
    """Собирает все результаты."""
    print(f"  [summary] Получено {len(state['deliverables'])} результатов")
    return {}

# --- Сборка графа ---
# Добавляем поле tasks в состояние для промежуточных данных
class FullProjectState(ProjectState):
    tasks: list[dict]

builder = StateGraph(FullProjectState)
builder.add_node("manager", manager)
builder.add_node("researcher", researcher)
builder.add_node("coder", coder)
builder.add_node("summary", summary)

builder.add_edge(START, "manager")
# Ключевой момент: ["researcher", "coder"] — два возможных целевых узла
builder.add_conditional_edges(
    "manager",
    dispatch_tasks,
    ["researcher", "coder"],
)
builder.add_edge("researcher", "summary")
builder.add_edge("coder", "summary")
builder.add_edge("summary", END)

app = builder.compile()
result = app.invoke(
    {"goal": "Создать чат-бота для поддержки клиентов", "deliverables": [], "tasks": []}
)
print(f"\n  Результаты ({len(result['deliverables'])}):")
for d in result["deliverables"]:
    print(f"  - {d[:100]}...")

  [manager] Задачи: [('research', 'Определить сценарии поддержки клиентов, '), ('code', 'Спроектировать и реализовать логику диал'), ('research', 'Выбрать подходящую платформу, API и инте')]

  Результаты (0):
